In [ ]:
import sys, torch
print("python:", sys.version.split()[0])          # must be >= 3.12 (see pyproject requires-python)
print("cuda  :", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert sys.version_info >= (3, 12), "epibudget requires Python >= 3.12; resolve before continuing"
assert torch.cuda.is_available(), "no GPU assigned; set Runtime -> T4 GPU"

!git clone --branch audit/scientific-hardening https://github.com/VivienP/epistasis-budget.git
%cd epistasis-budget
!git rev-parse HEAD                                # pin: record this commit alongside the result
!pip install -q -e .
!python scripts/fetch_gb1.py                       # GB1 (Wu 2016) -> data/proteingym/gb1_wu2016.csv

python: 3.12.13
cuda  : True NVIDIA A100-SXM4-80GB
Cloning into 'epistasis-budget'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (334/334), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 334 (delta 171), reused 286 (delta 123), pack-reused 0 (from 0)
Receiving objects: 100% (334/334), 225.28 KiB | 16.09 MiB/s, done.
Resolving deltas: 100% (171/171), done.
/content/epistasis-budget
3ba75ebbe700247654c824627fa98c4b8ba4010c
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for epibudget (pyproject.toml) ... done
  saved   data/proteingym/gb1_wu2016.csv  (10,934,877 bytes)
  sha256  2f115d4eaf03b6083dcc22f7451b3ddfad41c9d8e519286c4e69b6d06db78f1c
  rows    149,361
  orders  {0: 1, 1: 76, 2: 2091, 3: 26019, 4: 121174}
  pr

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/epibudget

Mounted at /content/drive


In [ ]:
!epibudget validate \
    --model esm2_t33_650M \
    --alphabet ADEF \
    --budgets 48 \
    --seeds 2 \
    --n-perturbations 2 \
    --device cuda \
    --batch-size 128 \
    --scored-cache /content/drive/MyDrive/epibudget/scored_smoke.jsonl \
    --out report_smoke/

validate scoring 307 candidates (alphabet='ADEF') with esm2_t33_650M on cuda …
config.json: 100% 724/724 [00:00<00:00, 3.93MB/s]
tokenizer_config.json: 100% 95.0/95.0 [00:00<00:00, 544kB/s]
vocab.txt: 100% 93.0/93.0 [00:00<00:00, 579kB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 558kB/s]
model.safetensors: 100% 2.61G/2.61G [00:18<00:00, 144MB/s]
Loading weights: 100% 539/539 [00:00<00:00, 19169.12it/s]
Var[ε] = 2.4217  [PASS invariant #1]   candidates=307  alphabet='ADEF'  
truth_terms=132
               recovery — pairwise (Spearman / Pearson, coverage)               
┏━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃  B ┃         info ┃      fitness ┃   structural ┃      random ┃     practice ┃
┡━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ 48 │ +0.613/+0.6… │ +0.575/+0.6… │ +0.595/+0.5… │ +0.335/+0.… │ +0.422/+0.4… │
│    │        c0.84 │        c0.83 │        c1.00 │       c0.66 │        c0.78 │
└────┴──

In [ ]:
import json
!python scripts/bench_scoring.py --model esm2_t33_650M --device cuda \
    --alphabet ACD --n-perturbations 16 --batch-size 128 --out report_smoke/bench_gpu.json
b = json.load(open("report_smoke/bench_gpu.json"))
vps = b["optimized_variants_per_s"]
eta_h = 29678 / vps / 3600
print(f"measured {vps:.3f} variants/s at n_perturbations=16  ->  full pool ETA ~ {eta_h:.1f} h")

Loading weights: 100% 539/539 [00:00<00:00, 11692.36it/s]
=== scoring throughput benchmark ===
  model_id                  : facebook/esm2_t33_650M_UR50D
  device                    : cuda
  alphabet                  : ACD
  max_order                 : 3
  n_perturbations           : 16
  batch_size                : 128
  num_threads               : None
  n_variants                : 137
  planned_rows              : 5848
  unique_rows               : 5631
  dedup_ratio               : 1.039
  reference_seconds         : 200.815
  optimized_seconds         : 140.367
  reference_variants_per_s  : 0.682
  optimized_variants_per_s  : 0.976
  speedup                   : 1.431
  max_abs_delta_g_gap       : 1.0013580322265625e-05
  max_abs_var_delta_g_gap   : 9.583548580938128e-06
  wrote report_smoke/bench_gpu.json
measured 0.976 variants/s at n_perturbations=16  ->  full pool ETA ~ 8.4 h


In [ ]:
import time
t0 = time.perf_counter()
!epibudget validate \
    --model esm2_t33_650M \
    --alphabet ACDEFGHIKLMNPQRSTVWY \
    --budgets 48,96,192 \
    --seeds 20 \
    --n-perturbations 16 \
    --device cuda \
    --batch-size 128 \
    --scored-cache /content/drive/MyDrive/epibudget/scored_650m.jsonl \
    --out report/
print(f"elapsed this cell: {(time.perf_counter() - t0) / 3600:.2f} h")

validate scoring 29678 candidates (alphabet='ACDEFGHIKLMNPQRSTVWY') with 
esm2_t33_650M on cuda …
config.json: 100% 724/724 [00:00<00:00, 4.36MB/s]
tokenizer_config.json: 100% 95.0/95.0 [00:00<00:00, 656kB/s]
vocab.txt: 100% 93.0/93.0 [00:00<00:00, 567kB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 960kB/s]
model.safetensors: 100% 2.61G/2.61G [00:07<00:00, 352MB/s]
Loading weights: 100% 539/539 [00:00<00:00, 6376.55it/s]
Var[ε] = 2.6162  [PASS invariant #1]   candidates=29678  
alphabet='ACDEFGHIKLMNPQRSTVWY'  truth_terms=17782
               recovery — pairwise (Spearman / Pearson, coverage)               
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃   B ┃         info ┃     fitness ┃   structural ┃      random ┃     practice ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│  48 │ +0.408/+0.4… │ -0.259/-0.… │ +0.484/+0.5… │ +0.279/+0.… │ -0.271/-0.2… │
│     │        c0.87 │       c0.06 │        c0.95 │

In [ ]:
import json, glob
from google.colab import files

# Find the most recent report generated under report/
path = sorted(glob.glob("report/*/metrics.json"))[-1]

# Print the configuration and Spearman values in the Colab console
m = json.load(open(path))
print("Fichier trouvé :", path)
print("Configuration :", {k: m.get(k) for k in ["device", "n_candidates", "n_perturbations", "candidate_alphabet", "seeds"]})

for r in m["results"]:
    pw = next((x for x in r["metrics"] if x["order"] == "pairwise"), None)
    if pw:
        print(f'  B={r["budget"]:>3} {r["method"]:<10} pairwise spearman={pw["spearman"]}')

# Trigger the automatic download of the file to the local machine
files.download(path)

Fichier trouvé : report/20260711T091947Z/metrics.json
Configuration : {'device': 'cuda', 'n_candidates': 29678, 'n_perturbations': 16, 'candidate_alphabet': 'ACDEFGHIKLMNPQRSTVWY', 'seeds': 20}
  B= 48 info       pairwise spearman=0.4081551885629586
  B= 48 fitness    pairwise spearman=-0.2590490176916245
  B= 48 structural pairwise spearman=0.48448676698450566
  B= 48 practice   pairwise spearman=-0.2709300398322422
  B= 48 random     pairwise spearman=0.27909976686693233
  B= 96 info       pairwise spearman=0.41826570123426604
  B= 96 fitness    pairwise spearman=-0.2471501239015881
  B= 96 structural pairwise spearman=0.4601914797445085
  B= 96 practice   pairwise spearman=0.28213953792031043
  B= 96 random     pairwise spearman=0.27954832076524827
  B=192 info       pairwise spearman=0.4430593471954884
  B=192 fitness    pairwise spearman=-0.1342133819845606
  B=192 structural pairwise spearman=0.5041874729834025
  B=192 practice   pairwise spearman=0.3048376417243201
  B=192 rando

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>